## LLM Architecture Configuration
We configure our model parameters to align with a standard autoregressive language modeling structure:
* **Vocabulary Size (`vocab_size`):** `30,000`
* **Hidden Dimension Size (`hidden_size`):** `768`
* **Causal Decoder Mode (`is_decoder`):** Set to `True` to apply causal (left-to-right) attention masking. This allows us to keep the target dimensions `(8, 30k) -> (8, 768)` consistent without conflict from BERT's bidirectional pre-training defaults.

## Step 1: Tokenization
We begin by preparing our raw input text string through sequence tokenization.

* **Training Sequence:** `"The bank of the river flooded"`

In [6]:
import torch
from transformers import AutoTokenizer, BertConfig, BertLMHeadModel

# Define the training text sequence
sentence = "The bank of the river flooded"

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Configure BERT as a causal decoder to enforce left-to-right attention masking
config = BertConfig.from_pretrained("bert-base-uncased")
config.is_decoder = True

# Load the pre-trained BERT model with the causal language modeling head
model = BertLMHeadModel.from_pretrained("bert-base-uncased", config=config)

# Tokenize input sequence with fixed-length padding and truncation
inputs = tokenizer(sentence, max_length=8, padding="max_length", truncation=True, return_tensors="pt")
print("Tokenized Inputs:", inputs)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenized Inputs: {'input_ids': tensor([[  101,  1996,  2924,  1997,  1996,  2314, 10361,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


## Step 2: Causal Modeling & Inference
We pass the tokenized sequence through the model to obtain predictions (logits) and compute the corresponding language modeling loss.

In [7]:
# Forward pass to calculate causal LM loss and logits
# Note: Hugging Face models internally shift labels to align next-token predictions
outputs = model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
logits = outputs.logits  # Shape: [Batch, Sequence_Length, Vocab_Size]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


## Step 3: Extracting Hidden States (Embeddings)
To feed representations into our custom attention layers, we extract the raw `768`-dimensional hidden state activations from the final layer of the transformer backplane, bypassing the vocabulary projection head.

In [8]:
# Forward pass returning hidden states of the transformer encoder layers
outputs_with_hidden = model(**inputs, output_hidden_states=True)

# Extract raw representation embeddings from the last decoder layer
# Shape: [Batch, Sequence_Length, Hidden_Size] (e.g., [1, 8, 768])
last_hidden_state = outputs_with_hidden.hidden_states[-1]
print(f"Extracted Hidden State Shape (Embeddings): {last_hidden_state.shape}")

# Assign embeddings to a clean descriptor for downstream layers
x_embeddings = last_hidden_state

Extracted Hidden State Shape (Embeddings): torch.Size([1, 8, 768])


## Step 4: Incorporating Positional Encodings
Since self-attention lacks inherent order-awareness, we inject sequential context by adding sinusoidal positional encodings to our extracted token embeddings prior to feeding them through our attention layers.

In [9]:
import numpy as np

def get_positional_encoding(seq_len: int, d_model: int) -> torch.Tensor:
    """Generates a sinusoidal positional encoding tensor."""
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0)  # Add batch dimension shape: [1, seq_len, d_model]

# Generate positional encodings matched to our sequence length (8) and embedding dimension (768)
B, T, D = x_embeddings.shape
pos_encoding = get_positional_encoding(T, D).to(x_embeddings.device)

# Combine the embeddings and positional encodings
x_with_pos = x_embeddings + pos_encoding

print(f"Positional Encoding Shape:  {pos_encoding.shape}")
print(f"Position-Aware Input Shape: {x_with_pos.shape}")

Positional Encoding Shape:  torch.Size([1, 8, 768])
Position-Aware Input Shape: torch.Size([1, 8, 768])


## Step 5: Layer Normalization
Unlike **Batch Normalization** which normalizes across the batch dimension for each feature column individually (and can be unstable with dynamic sequence lengths), **Layer Normalization** normalizes across the feature dimension (the row of a single data sample).

In Large Language Models (LLMs) and Transformer architectures, Layer Normalization is critical because:
* It stabilizes the hidden state dynamics across deep layers.
* It acts independently on each sequence token, making it highly compatible with variable-length inputs and autoregressive generation.

Below, we apply our Custom Layer Normalization implementation to the position-aware embeddings tensor (`x_with_pos`) prior to feeding it into our Multi-Head Attention layer.

In [10]:
import torch
import torch.nn as nn

class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        # Learnable parameters: gamma (initialized to 1), beta (initialized to 0)
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, unbiased=False, keepdim=True)  # MLE variance
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

# Initialize LayerNorm with BERT's hidden dimension size (768)
layer_norm = CustomLayerNorm(normalized_shape=768).to(x_with_pos.device)

# Apply Layer Normalization to the position-aware embeddings
x_norm = layer_norm(x_with_pos)

print(f"Original Embedding Shape:      {x_with_pos.shape}")
print(f"Layer-Normalized Shape:        {x_norm.shape}")

Original Embedding Shape:      torch.Size([1, 8, 768])
Layer-Normalized Shape:        torch.Size([1, 8, 768])


## Step 6: Custom Multi-Head Self-Attention Integration
We define our custom `MultiHeadAttention` class with support for causal masking to prevent information leakage from future tokens. We then initialize the layer using BERT's default configuration parameters (`d_model=768` and `num_heads=12`).

In [11]:
# Multi-head Attention
# B: Batch Size (e.g., 1 sequence)
# T: Sequence Length / Time Steps (e.g., 8 tokens)
# D: Feature Dimension / Hidden Size / Vocab Size (e.g., 30522 logits)
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model:int, num_heads:int, dropout: float = 0.0):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    self.head_dim = self.d_k  # Add alias to prevent AttributeError in any version

    # (QKt/sq(model))* V = (d_model, d_model)
    self.qkv_proj = nn.Linear(d_model, 3 * d_model)
    self.out_proj = nn.Linear(d_model, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x:torch.Tensor, is_causal: bool = False)->torch.Tensor:
    B, T, D = x.shape
    qkv = self.qkv_proj(x)
    q,k,v = qkv.chunk(3, dim=-1)
    q,k,v = map(lambda t: t.view(B, T, self.num_heads, self.d_k).transpose(1,2), (q,k,v))
    attention_pattern = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)

    # Apply causal mask only if is_causal is True
    if is_causal:
      causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
      attention_pattern = attention_pattern.masked_fill(~causal_mask, float('-inf'))

    # 5. Softmax along key sequence dimension
    attn_weights = F.softmax(attention_pattern, dim=-1)
    attn_weights = self.dropout(attn_weights)
    # 6. Weighted sum over values: (B, H, T, T) @ (B, H, T, d_k) -> (B, H, T, d_k)
    context = torch.matmul(attn_weights, v)
    context = context.transpose(1, 2).contiguous().view(B, T, D)

    # 8. Final linear projection
    return self.out_proj(context)

## Step 7: Verification & Structural Evaluation
Finally, we feed our position-aware sequence embeddings (`x_with_pos`) through the initialized attention block with causal masking enabled, validating that structural dimensionality remains perfectly consistent.

In [12]:
# Instantiate Multi-Head Attention layer with BERT architecture parameters
# Model dimension (d_model) = 768, Number of heads = 12
mha_768 = MultiHeadAttention(d_model=768, num_heads=12)

# Process normalized position-aware embeddings using causal self-attention
attention_output = mha_768(x_norm, is_causal=True)

# Print and verify incoming/outgoing structural tensor dimensions
print(f"Layer-Normalized Input Shape: {x_norm.shape}")
print(f"Attention Output Shape:       {attention_output.shape}")

Layer-Normalized Input Shape: torch.Size([1, 8, 768])
Attention Output Shape:       torch.Size([1, 8, 768])


### Shape Verification & Comparison
Let's verify that the output of our custom Multi-Head Attention layer maintains the exact structural properties of our input embeddings, confirming that the token embeddings have been contextualized without shape distortion.

In [23]:
# Cell left empty or removed during consolidation.

In [13]:
# Compare shapes directly using assertions
assert x_norm.shape == attention_output.shape, f"Mismatch! Input shape {x_norm.shape} != Output shape {attention_output.shape}"

print("=== Dimensional Comparison ===")
print(f"Input Embeddings Shape:  {list(x_norm.shape)}  -> [Batch, Seq_Len, Hidden_Dim]")
print(f"Attention Output Shape:  {list(attention_output.shape)}  -> [Batch, Seq_Len, Hidden_Dim]")
print("\n✅ Success: The custom Multi-Head Attention layer successfully preserves the spatial and feature dimensions.")

=== Dimensional Comparison ===
Input Embeddings Shape:  [1, 8, 768]  -> [Batch, Seq_Len, Hidden_Dim]
Attention Output Shape:  [1, 8, 768]  -> [Batch, Seq_Len, Hidden_Dim]

✅ Success: The custom Multi-Head Attention layer successfully preserves the spatial and feature dimensions.


### Summary of Architecture Steps
1. **Tokenization:** Text parsed into token IDs (length = 8).
2. **Causal Modeling:** Forward pass through `BertLMHeadModel` to establish causal context.
3. **Extraction:** Extracted the $H=768$ representation matrix from the final decoder layer.
4. **Positional Injection:** Injected sequence-order awareness via sinusoidal positional encodings (`x_with_pos`).
5. **Layer Normalization:** Applied custom `CustomLayerNorm` on the position-aware embeddings to stabilize the hidden state dynamics (`x_norm`).
6. **Multi-Head Attention:** Passed normalized, position-aware contextual embeddings through a custom Multi-Head Causal Self-Attention layer to output a structurally matched representation.

In [25]:
# All steps completed and verified successfully in previous sections.